In [1]:
# load imports and data

import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw_adoptions.csv")

print(df.shape)
df.head()

(173775, 12)


,animal_id,date_of_birth,datetime,monthyear,outcome_type,outcome_subtype,animal_type,sex_upon_outcome,age_upon_outcome,breed,color,name
0,A668305,2012-12-01,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow,NaN
1,A673335,2012-02-22,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray,NaN
2,A675999,2013-04-03,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green,NaN
3,A679066,2014-04-16,2014-05-16T00:00:00-05:00,05-2014,NaN,NaN,Other,Unknown,4 weeks,Rabbit Sh,Brown,NaN
4,A680855,2014-05-25,2014-06-10T00:00:00-05:00,06-2014,Transfer,Partner,Bird,Unknown,2 weeks,Duck,Yellow/Black,NaN


In [2]:
# drop unnecessary cols

df = df.drop(columns=['monthyear', 'outcome_subtype'])

print(df.shape)
print(df.columns.tolist())

(173775, 10)
['animal_id', 'date_of_birth', 'datetime', 'outcome_type', 'animal_type', 'sex_upon_outcome', 'age_upon_outcome', 'breed', 'color', 'name']


In [7]:
# create binary target var
# 0 = not adopted: transfer, rto, euth, etc
# 1 = adopted: adoption, rto-adopt

# drop rows where outcome_type is missing
df = df.dropna(subset=['outcome_type'])

# create "adopted" binary var, group "adoption" and "rto-adopt" into "adopted"; all other outcomes are "not adopted"
df['adopted'] = df['outcome_type'].isin(['Adoption', 'Rto-Adopt']).astype(int)

# check value counts for new var (count and %)
print(df['adopted'].value_counts())
print(df['adopted'].value_counts(normalize=True).round(3) * 100)

adopted
0    87890
1    85839
Name: count, dtype: int64
adopted
0    50.6
1    49.4
Name: proportion, dtype: float64


In [8]:
# feature engineering: sex_upon_outcome
# split sex and fixed status into separate features

# create "is_fixed" binary feature
df['is_fixed'] = df['sex_upon_outcome'].isin(['Neutered Male', 'Spayed Female']).astype(int)

# extract sex
def extract_sex(value):
    if pd.isna(value):
        return 'Unknown'
    if 'Male' in value:
        return 'Male'
    elif 'Female' in value:
        return 'Female'
    else:
        return 'Unknown'

# create "sex" feature using extracted_sex
df['sex'] = df['sex_upon_outcome'].apply(extract_sex)

# check value counts for new features
print(df['is_fixed'].value_counts())
print(df['sex'].value_counts())

is_fixed
1    116176
0     57553
Name: count, dtype: int64
sex
Male       83178
Female     77055
Unknown    13496
Name: count, dtype: int64


In [9]:
# drop unnecessary sex_upon_outcome col

df = df.drop(columns=['sex_upon_outcome'])

print(df.columns.tolist())

['animal_id', 'date_of_birth', 'datetime', 'outcome_type', 'animal_type', 'age_upon_outcome', 'breed', 'color', 'name', 'adopted', 'is_fixed', 'sex']
